# Risk Agent — user perspective

**What it does.** Three skills, two execution modes:

- **Skills 12 + 13** (quality scoring + adaptive prioritization) — pure Python math. No LLM, no MCP. Always safe to run.
- **Skill 14** (auto-improvement trigger detection) — one isolated `query()` call with `allowed_tools=[]` and `mcp_servers=None` (Pattern C). The LLM physically cannot write to Linear.

**RISK-04 defence in depth.** No Linear writes by the agent itself. Layer 1 = empty allow-list; Layer 2 = `Literal["suggested"]` on `AutoImprovementTrigger.linear_state`; Layer 3 = no `linear_agent` import; Layer 4 = `linear_write_guard` on every write path.

**Entry points.**
- `RiskAgent().calculate_quality_score(work_item, qa_history, uat_results) -> QualityScore`
- `RiskAgent().get_priority_queue(ready_tasks, linear_state) -> PriorityQueue`
- `await RiskAgent().detect_improvement_triggers(qa_history, scores)`

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Quality score (skill 12)

Deterministic formula: `score = max(0, 100 - 10*qa_failures - 5*fix_subtasks - (15 if uat_failed else 0) - 5*rework_cycles)`. Risk levels: `>=75` low, `>=50` medium, else high. Always runs — no flags.

In [ ]:
from hsb.agents.risk_agent import RiskAgent

agent = RiskAgent()
work_item = {"id": "LIN-1", "fix_subtask_count": 1, "qa_cycle_count": 1}
qa_history = [{"status": "changes_required"}, {"status": "approved"}]
uat_results = [{"overall_status": "approved"}]

q = agent.calculate_quality_score(work_item, qa_history, uat_results)
print("score:    ", q.score)
print("risk:     ", RiskAgent.risk_level(q.score))
print("breakdown:", q.score_breakdown)

## Priority queue (skill 13)

Sorts ready tasks by score descending, with `updatedAt` ascending as the tiebreak. Pure Python — runs offline.

In [ ]:
ready = ["LIN-1", "LIN-2", "LIN-3"]
state = {
    "LIN-1": {"id": "LIN-1", "qa_cycle_count": 2, "updatedAt": "2026-05-01"},
    "LIN-2": {"id": "LIN-2", "qa_cycle_count": 0, "updatedAt": "2026-05-02"},
    "LIN-3": {"id": "LIN-3", "qa_cycle_count": 1, "updatedAt": "2026-05-03"},
}
pq = agent.get_priority_queue(ready, state)
for tid in pq.items:
    print(f"  {tid}: score={pq.scores[tid]}")

## Auto-improvement triggers (skill 14, gated)

The only LLM call in this agent. `allowed_tools=[]` is asserted before the call — see `risk_agent.py` for the structural guard. Gated.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("Risk skill 14 live run"))
else:
    assert_g1_safe()
    import asyncio

    triggers = asyncio.run(
        agent.detect_improvement_triggers(
            qa_history=[
                {
                    "work_item_id": "LIN-1",
                    "category": "architecture",
                    "summary": "drift",
                }
            ],
            scores=[q],
        )
    )
    print("triggers:", len(triggers))
    for t in triggers:
        print(f"  [{t.suggested_type}] {t.title}")